In [2]:
from datasets import load_dataset

dataset = load_dataset("lamini/earnings-calls-qa", split="train")

/home/srirang/miniconda3/envs/graphrag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 860164/860164 [00:49<00:00, 17518.99 examples/s]


In [3]:
print(type(dataset))

<class 'datasets.arrow_dataset.Dataset'>


In [4]:
dataset.column_names

['question', 'answer', 'date', 'transcript', 'q', 'ticker', 'predictions']

In [5]:
dataset

Dataset({
    features: ['question', 'answer', 'date', 'transcript', 'q', 'ticker', 'predictions'],
    num_rows: 860164
})

In [6]:
import pandas as pd

In [7]:
df = dataset.to_pandas()

In [8]:
# Keep the corpus within one tightly connected industry so GraphRAG has
# clear multi-hop advantages over standard RAG.
semis = [
    "ADI",
    "AMD",
    "AMAT",
    "ASML",
    "AVGO",
    "INTC",
    "KLAC",
    "LRCX",
    "MRVL",
    "MU",
    "NVDA",
    "NXPI",
    "ON",
    "QCOM",
    "TSM",
    "TXN"
]

filtered = df[df["ticker"].isin(semis)].copy()

# Drop obviously unusable rows before grouping.
filtered = filtered[
    filtered["transcript"].notna() &
    filtered["question"].notna() &
    filtered["answer"].notna() &
    filtered["date"].notna() &
    filtered["q"].notna()
]

In [9]:
filtered

,question,answer,date,transcript,q,ticker,predictions
0,What was TSMC's revenue in US dollar terms in ...,I do not know. The transcript does not provid...,"Jan 13, 2022, 1:00 a.m. ET",and for our industry-leading advanced and spec...,2021-Q4,TSM,"[{'class_id': 0, 'class_name': 'correct', 'pro..."
1,What was TSMC's EPS in 2019,I do not know. The transcript does not provid...,"Jan 13, 2022, 1:00 a.m. ET",and for our industry-leading advanced and spec...,2021-Q4,TSM,"[{'class_id': 0, 'class_name': 'correct', 'pro..."
2,What was TSMC's capex spending in 2019,I do not know. The transcript does not provid...,"Jan 13, 2022, 1:00 a.m. ET",and for our industry-leading advanced and spec...,2021-Q4,TSM,"[{'class_id': 0, 'class_name': 'correct', 'pro..."
3,What is the expected growth rate of global sma...,The expected growth rate of global smartphone...,"Jan 14, 2021, 1:00 a.m. ET",g demand for our advanced technologies in the ...,2020-Q4,TSM,"[{'class_id': 0, 'class_name': 'correct', 'pro..."
4,What is the expected penetration rate for 5G s...,The expected penetration rate for 5G smartpho...,"Jan 14, 2021, 1:00 a.m. ET",g demand for our advanced technologies in the ...,2020-Q4,TSM,"[{'class_id': 0, 'class_name': 'correct', 'pro..."
...,...,...,...,...,...,...,...
857972,What is the company's plan to drive to the gro...,The company's plan to drive to the gross marg...,"May 11, 2020, 9:00 a.m. ET",guys. Thanks for letting me ask a question. I...,2020-Q1,ON,"[{'class_id': 0, 'class_name': 'correct', 'pro..."
859726,What was the company's operating income for th...,I do not know. The transcript does not provid...,"Mar 03, 2021, 4:45 p.m. ET",Analyst\nMore MRVL analysis\nAll earnings call...,2021-Q4,MRVL,"[{'class_id': 0, 'class_name': 'correct', 'pro..."
860098,What is the company's revenue growth rate for ...,The company's revenue growth rate for the sec...,"May 03, 2022, 8:00 a.m. ET",ere again because you see we are still below t...,2022-Q1,NXPI,"[{'class_id': 0, 'class_name': 'correct', 'pro..."
860099,What is the company's revenue growth rate for ...,The company's revenue growth rate for the fir...,"May 03, 2022, 8:00 a.m. ET",ere again because you see we are still below t...,2022-Q1,NXPI,"[{'class_id': 0, 'class_name': 'correct', 'pro..."


In [10]:
import numpy as np

In [11]:
grouped_calls = filtered.groupby(["ticker", "q", "date"], as_index=False).agg({
    "question": list,
    "answer": list,
    "transcript": list,
    "predictions": list
})

In [12]:
import json
import re
from pathlib import Path

output_dir = Path("data/raw/earnings_calls")
output_dir.mkdir(parents=True, exist_ok=True)

def unique_non_empty(values):
    seen = set()
    result = []

    for value in values:
        if not isinstance(value, str):
            continue

        cleaned = value.strip()
        if not cleaned or cleaned in seen:
            continue

        seen.add(cleaned)
        result.append(cleaned)

    return result
def to_json_safe(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, list):
        return [to_json_safe(item) for item in value]
    if isinstance(value, dict):
        return {key: to_json_safe(val) for key, val in value.items()}
    return value

for _, row in grouped_calls.iterrows():

    qa_pairs = []
    for question, answer, predictions in zip(row["question"], row["answer"], row["predictions"]):
        qa_pairs.append({
            "question": question,
            "answer": answer,
            "predictions": to_json_safe(predictions)
        })

    transcript_snippets = unique_non_empty(row["transcript"])
    combined_transcript = "\n\n".join(transcript_snippets)

    safe_date = re.sub(r"[^A-Za-z0-9._-]+", "_", row["date"][:20]).strip("_")
    filename = f"{row['ticker']}_{row['q']}_{safe_date}.json"

    data = {
        "ticker": row["ticker"],
        "quarter": row["q"],
        "date": row["date"],
        "num_qa_pairs": len(qa_pairs),
        "num_transcript_snippets": len(transcript_snippets),
        "qa_pairs": qa_pairs,
        "transcript_snippets": transcript_snippets,
        "combined_transcript": combined_transcript
    }

    with open(output_dir / filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

In [13]:
grouped_calls.shape

(81, 7)

In [14]:
grouped_calls.iloc[0][["ticker", "q", "date"]]

ticker                            ADI
q                             2021-Q1
date      Feb 17, 2021, 10:00 a.m. ET
Name: 0, dtype: object

In [15]:
from pathlib import Path

files = sorted(Path("data/raw/earnings_calls").glob("*.json"))
len(files), files[:3]

(81,
 [PosixPath('data/raw/earnings_calls/ADI_2021-Q1_Feb_17_2021_10_00.json'),
  PosixPath('data/raw/earnings_calls/ADI_2021-Q2_May_19_2021_10_00.json'),
  PosixPath('data/raw/earnings_calls/ADI_2021-Q3_Aug_18_2021_10_00.json')])

In [16]:
import json

sample_file = files[0]
with open(sample_file, "r", encoding="utf-8") as f:
    sample = json.load(f)

sample.keys()

dict_keys(['ticker', 'quarter', 'date', 'num_qa_pairs', 'num_transcript_snippets', 'qa_pairs', 'transcript_snippets', 'combined_transcript'])

In [17]:
sample["ticker"], sample["quarter"], sample["num_qa_pairs"], sample["num_transcript_snippets"]

('ADI', '2021-Q1', 58, 20)

In [ ]:
import json
from pathlib import Path

files = sorted(Path("data/raw/earnings_calls").glob("*.json"))

char_total = 0
qa_total = 0

for p in files:
    obj = json.loads(p.read_text(encoding="utf-8"))
    char_total += len(obj.get("combined_transcript", ""))
    qa_total += obj.get("num_qa_pairs", 0)

print("files:", len(files))
print("combined chars:", char_total)
print("rough token estimate:", char_total // 4)
print("qa pairs:", qa_total)

files: 81
combined chars: 8188763
rough token estimate: 2047190
qa pairs: 7146
